[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione-superiori/blob/main/03_lo_scout_migliora.ipynb)

# Notebook 3 — Lo scout migliora

Uno scout umano non giudica un calciatore guardando un solo numero. In questo episodio passiamo da una regressione con una variabile a una regressione con più variabili.

In [ ]:
from IPython.display import HTML, display

def concept(title, body, kind="idea"):
    colors = {
        "idea": ("#e8f3ff", "#0b5394"),
        "math": ("#fff3cd", "#7a4d00"),
        "warning": ("#fdecea", "#a61b1b"),
        "task": ("#eaf7ea", "#226622"),
        "story": ("#f3e8ff", "#5b2a86")
    }
    bg, border = colors.get(kind, colors["idea"])
    display(HTML(f'''
    <div style="background:{bg}; border-left:6px solid {border}; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
    <b style="color:{border}; font-size:18px">{title}</b><br>{body}
    </div>
    '''))

def reveal(title, body):
    display(HTML(f'''
    <details style="background:#f7f7f7; border:1px solid #ddd; padding:12px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.45">
      <summary style="cursor:pointer; font-weight:bold">{title}</summary>
      <div style="margin-top:10px">{body}</div>
    </details>
    '''))

In [ ]:
concept("Missione", "Confrontare un modello semplice con un modello piu' informato, usando age, overall, potential e wage_eur.", "story")

In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = list(DATA_DIR.rglob("players_22.csv")) or list(DATA_DIR.rglob("*players*.csv"))
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Dataset caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
concept("Regressione multipla", "Nella regressione multipla il modello usa piu' input contemporaneamente. Geometricamente non stiamo piu' adattando una retta nel piano, ma un piano o un iperpiano in uno spazio con piu' dimensioni. L'idea resta la stessa: trovare la combinazione di variabili che riduce l'errore.", "idea")

<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Regressione lineare multipla</b>

<p>Con $p$ variabili di input $x_1, x_2, \dots, x_p$, il modello diventa:</p>

$$\hat{y} \;=\; w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b$$

<p>Ogni variabile ha il suo peso $w_j$. In forma compatta, se $\mathbf{x} = (x_1, \dots, x_p)$ e $\mathbf{w} = (w_1, \dots, w_p)$:</p>

$$\hat{y} \;=\; \mathbf{w}^{\top}\mathbf{x} + b$$

<p>Geometricamente, con $1$ variabile cerchiamo una <b>retta</b>; con $2$ un <b>piano</b>; con $p$ un <b>iperpiano</b> in $p$ dimensioni. La logica resta la stessa: scegliere $\mathbf{w}$ e $b$ che minimizzano la stessa funzione di costo del notebook precedente:</p>

$$\mathrm{MSE}(\mathbf{w}, b) \;=\; \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
</div>


## Modello A: solo overall

In [ ]:
X_one = df[["overall"]]
y = df["value_mln_eur"]

model_one = LinearRegression()
model_one.fit(X_one, y)
pred_one = model_one.predict(X_one)

mae_one = mean_absolute_error(y, pred_one)
rmse_one = mean_squared_error(y, pred_one) ** 0.5
r2_one = r2_score(y, pred_one)

print(f"MAE modello A: {mae_one:.2f} milioni")
print(f"RMSE modello A: {rmse_one:.2f} milioni")
print(f"R^2 modello A: {r2_one:.3f}")

## Modello B: più indizi

In [ ]:
features = [c for c in ["age", "overall", "potential", "wage_eur"] if c in df.columns]
X_many = df[features]

model_many = LinearRegression()
model_many.fit(X_many, y)
pred_many = model_many.predict(X_many)

mae_many = mean_absolute_error(y, pred_many)
rmse_many = mean_squared_error(y, pred_many) ** 0.5
r2_many = r2_score(y, pred_many)

print("Feature usate:", features)
print(f"MAE modello B: {mae_many:.2f} milioni")
print(f"RMSE modello B: {rmse_many:.2f} milioni")
print(f"R^2 modello B: {r2_many:.3f}")

In [ ]:
concept("Confronto tra modelli", "Un modello piu' ricco puo' usare piu' informazione. Se aggiungiamo variabili sensate, l'errore tende a diminuire. Pero' aggiungere variabili non e' sempre una vittoria: alcune colonne possono essere rumorose, ridondanti o persino fuorvianti.", "warning")

## Confrontiamo gli errori

In [ ]:
comparison = pd.DataFrame({
    "modello": ["A: solo overall", "B: piu' indizi"],
    "MAE_mln": [mae_one, mae_many],
    "RMSE_mln": [rmse_one, rmse_many],
    "R2": [r2_one, r2_many]
})
comparison.round(3)

## Quanto pesa ogni variabile?

In [ ]:
coef = pd.DataFrame({"feature": features, "coefficiente": model_many.coef_})
coef.sort_values("coefficiente", ascending=False)

In [ ]:
concept("Attenzione ai coefficienti", "I coefficienti non sono sempre confrontabili direttamente, perche' le variabili hanno scale diverse. Un punto di overall e un euro di salario non hanno la stessa unita'. Per interpretazioni serie bisognerebbe normalizzare le variabili. Qui usiamo i coefficienti solo per intuire che il modello combina piu' indizi.", "warning")

## Confronto onesto dei pesi: standardizzare le variabili


<div style="background:#fff3cd; border-left:6px solid #7a4d00; padding:14px; margin:12px 0; border-radius:8px; font-size:16px; line-height:1.5">
<b style="color:#7a4d00; font-size:18px">Teoria &mdash; Standardizzare per confrontare i pesi</b>

<p>Le nostre variabili hanno <b>scale molto diverse</b>: l'overall va da circa $50$ a $95$, il salario da migliaia a centinaia di migliaia di euro. Il coefficiente del salario &egrave; numericamente piccolo solo perch&eacute; il salario &egrave; un numero grande, non perch&eacute; il salario "conti poco".</p>

<p>Per rendere i pesi confrontabili, trasformiamo ogni variabile cos&igrave;:</p>

$$z_j \;=\; \frac{x_j - \bar{x}_j}{\sigma_j}$$

<p>dove $\bar{x}_j$ &egrave; la media e $\sigma_j$ la deviazione standard della $j$-esima variabile. Dopo la trasformazione, tutte le variabili hanno <b>media $0$ e deviazione standard $1$</b>: ora il coefficiente pi&ugrave; grande in valore assoluto indica la variabile pi&ugrave; influente.</p>
</div>


In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardizziamo le feature: media 0, deviazione standard 1 per ognuna
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_many)

model_scaled = LinearRegression()
model_scaled.fit(X_scaled, y)

# Ora i pesi sono confrontabili
pesi = pd.DataFrame({
    "feature": features,
    "peso_standardizzato": model_scaled.coef_
}).sort_values("peso_standardizzato", key=lambda s: s.abs(), ascending=False)
pesi.round(3)


In [ ]:
# Visualizziamo i pesi standardizzati
plt.figure(figsize=(7,4))
colori = ["#2a9d8f" if w > 0 else "#e76f51" for w in pesi["peso_standardizzato"]]
plt.barh(pesi["feature"], pesi["peso_standardizzato"], color=colori)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Peso (su variabili standardizzate)")
plt.title("Quale variabile pesa di piu' sul valore di mercato?")
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis="x")
plt.show()


## Lo scout automatico su giocatori casuali

In [ ]:
sample = df.sample(12, random_state=13).copy()
sample["pred_solo_overall"] = model_one.predict(sample[["overall"]])
sample["pred_piu_indizi"] = model_many.predict(sample[features])
sample[["short_name", "age", "overall", "potential", "wage_eur", "value_mln_eur", "pred_solo_overall", "pred_piu_indizi"]].round(2)

In [ ]:
reveal("Domanda", "Trovate un giocatore per cui il modello con piu' indizi migliora molto rispetto al modello con solo overall. Quale informazione extra probabilmente lo ha aiutato?")

In [ ]:
reveal("Cosa dovremmo aver capito", "La regressione non e' solo una retta: e' un modo per costruire una funzione che trasforma informazioni in una previsione numerica. Nel prossimo episodio faremo il controllo piu' importante: il modello funziona anche su giocatori che non ha usato per imparare?")